In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

meta = pd.read_csv("../data/ISIC_2019_Training_Metadata.csv")
labels = pd.read_csv("../data/ISIC_2019_Training_GroundTruth.csv")

# Join on isic_id = image
df = meta.merge(labels, left_on="isic_id", right_on="image")

print("Total rows:", len(df))
print("Total columns:", len(df.columns))
print("\nAll columns:")
for col in df.columns:
    print(f"  {col}")

Total rows: 23257
Total columns: 37

All columns:
  isic_id
  attribution
  copyright_license
  age_approx
  anatom_site_1
  anatom_site_2
  anatom_site_3
  anatom_site_4
  anatom_site_5
  anatom_site_general
  anatom_site_special
  clin_size_long_diam_mm
  concomitant_biopsy
  dermoscopic_type
  diagnosis_1
  diagnosis_2
  diagnosis_3
  diagnosis_4
  diagnosis_5
  diagnosis_confirm_type
  family_hx_mm
  image_type
  lesion_id
  melanocytic
  patient_id
  personal_hx_mm
  sex
  image
  MEL
  NV
  BCC
  AK
  BKL
  DF
  VASC
  SCC
  UNK


In [4]:
print("="*60)
print("FIELD AUDIT — dtype, nulls, unique values")
print("="*60)

for col in df.columns:
    nulls = df[col].isnull().sum()
    null_pct = round(nulls / len(df) * 100, 1)
    unique = df[col].nunique()
    dtype = df[col].dtype
    print(f"\n[{col}]")
    print(f"  dtype   : {dtype}")
    print(f"  nulls   : {nulls} ({null_pct}%)")
    print(f"  unique  : {unique}")
    if unique <= 15 and dtype == object:
        print(f"  values  : {df[col].value_counts().to_dict()}")
    elif dtype in ['float64', 'int64']:
        print(f"  range   : {df[col].min()} – {df[col].max()}")
        print(f"  mean    : {round(df[col].mean(), 2)}")

FIELD AUDIT — dtype, nulls, unique values

[isic_id]
  dtype   : object
  nulls   : 0 (0.0%)
  unique  : 23257

[attribution]
  dtype   : object
  nulls   : 0 (0.0%)
  unique  : 3
  values  : {'Hospital Clínic de Barcelona': 12413, 'ViDIR Group, Department of Dermatology, Medical University of Vienna': 10015, 'Anonymous': 829}

[copyright_license]
  dtype   : object
  nulls   : 0 (0.0%)
  unique  : 2
  values  : {'CC-BY-NC': 22428, 'CC-0': 829}

[age_approx]
  dtype   : float64
  nulls   : 268 (1.2%)
  unique  : 18
  range   : 0.0 – 85.0
  mean    : 54.42

[anatom_site_1]
  dtype   : object
  nulls   : 878 (3.8%)
  unique  : 5
  values  : {'Trunk': 10559, 'Lower extremity': 4737, 'Head and neck': 4386, 'Upper extremity': 2649, 'Anogenital region': 48}

[anatom_site_2]
  dtype   : object
  nulls   : 12490 (53.7%)
  unique  : 17

[anatom_site_3]
  dtype   : object
  nulls   : 20707 (89.0%)
  unique  : 13
  values  : {'Anterior abdomen': 1114, 'Face': 751, 'Anterior chest': 483, 'Scalp': 

In [5]:
import sys

with open("../data/field_audit.txt", "w", encoding="utf-8") as f:
    f.write(f"Total rows: {len(df)}\n")
    f.write(f"Total columns: {len(df.columns)}\n\n")
    f.write("="*60 + "\n")
    f.write("FIELD AUDIT\n")
    f.write("="*60 + "\n")
    
    for col in df.columns:
        nulls = df[col].isnull().sum()
        null_pct = round(nulls / len(df) * 100, 1)
        unique = df[col].nunique()
        dtype = df[col].dtype
        f.write(f"\n[{col}]\n")
        f.write(f"  dtype   : {dtype}\n")
        f.write(f"  nulls   : {nulls} ({null_pct}%)\n")
        f.write(f"  unique  : {unique}\n")
        if unique <= 20 and dtype == object:
            f.write(f"  values  : {df[col].value_counts().to_dict()}\n")
        elif dtype in ['float64', 'int64']:
            f.write(f"  range   : {df[col].min()} – {df[col].max()}\n")
            f.write(f"  mean    : {round(df[col].mean(), 2)}\n")

print("Saved to C:\\LesionIQ\\data\\field_audit.txt")

Saved to C:\LesionIQ\data\field_audit.txt


In [6]:
# Define final feature columns
FEATURES = ['age_approx', 'sex', 'anatom_site_general', 'anatom_site_1']
LABELS = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
AUDIT = ['diagnosis_1', 'diagnosis_2', 'diagnosis_3', 'diagnosis_confirm_type',
         'concomitant_biopsy', 'melanocytic']
LEAKAGE_GUARD = ['lesion_id']

df_clean = df[['isic_id'] + FEATURES + LABELS + AUDIT + LEAKAGE_GUARD].copy()

# Class counts
label_counts = df[LABELS].sum().sort_values(ascending=False).astype(int)
print("Class distribution:")
print(label_counts)
print(f"\nRare classes (< 5% of data): {label_counts[label_counts < 0.05 * len(df)].index.tolist()}")

df_clean.to_csv("../data/layer0_clean.csv", index=False)
print(f"\nSaved: layer0_clean.csv — {len(df_clean)} rows, {len(df_clean.columns)} columns")

Class distribution:
NV      11559
MEL      4148
BCC      3323
BKL      2240
AK        867
SCC       628
VASC      253
DF        239
dtype: int32

Rare classes (< 5% of data): ['AK', 'SCC', 'VASC', 'DF']

Saved: layer0_clean.csv — 23257 rows, 20 columns


In [7]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Work on a copy
df_model = df_clean.copy()

# --- age_approx: impute with per-class median ---
# First get the dominant class label per row
label_cols = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
df_model['class'] = df_model[label_cols].idxmax(axis=1)

age_medians = df_model.groupby('class')['age_approx'].median()
print("Age median per class:")
print(age_medians.round(1))

df_model['age_approx'] = df_model.apply(
    lambda row: age_medians[row['class']] if pd.isnull(row['age_approx']) else row['age_approx'],
    axis=1
)

# --- sex: fill nulls as 'unknown', then one-hot encode ---
df_model['sex'] = df_model['sex'].fillna('unknown')
sex_dummies = pd.get_dummies(df_model['sex'], prefix='sex')
print("\nSex categories:", sex_dummies.columns.tolist())

# --- anatom_site_general: fill nulls as 'unknown', then one-hot encode ---
df_model['anatom_site_general'] = df_model['anatom_site_general'].fillna('unknown')
site_dummies = pd.get_dummies(df_model['anatom_site_general'], prefix='site')
print("Site categories:", site_dummies.columns.tolist())

# --- Combine everything ---
df_model = pd.concat([df_model[['isic_id','age_approx','class'] + label_cols], 
                       sex_dummies, site_dummies], axis=1)

print(f"\nFinal model-ready shape: {df_model.shape}")
print(f"Null check:\n{df_model.isnull().sum()[df_model.isnull().sum() > 0]}")

df_model.to_csv("../data/layer0_model_ready.csv", index=False)
print("\nSaved: layer0_model_ready.csv")

Age median per class:
class
AK      70.0
BCC     70.0
BKL     65.0
DF      50.0
MEL     60.0
NV      45.0
SCC     70.0
VASC    55.0
Name: age_approx, dtype: float64

Sex categories: ['sex_female', 'sex_male', 'sex_unknown']
Site categories: ['site_anterior torso', 'site_head/neck', 'site_lateral torso', 'site_lower extremity', 'site_oral/genital', 'site_palms/soles', 'site_posterior torso', 'site_unknown', 'site_upper extremity']

Final model-ready shape: (23257, 23)
Null check:
Series([], dtype: int64)

Saved: layer0_model_ready.csv


In [8]:
from sklearn.model_selection import train_test_split

# Reload with lesion_id for split guarding
df_split = df_model.copy()
df_split['lesion_id'] = df_clean['lesion_id']

# Images with no lesion_id get a unique placeholder so they don't group together
df_split['lesion_id'] = df_split['lesion_id'].fillna(df_split['isic_id'])

# Get unique lesion IDs and split those — not individual images
unique_lesions = df_split['lesion_id'].unique()
train_lesions, temp_lesions = train_test_split(unique_lesions, test_size=0.2, random_state=42)
val_lesions, test_lesions = train_test_split(temp_lesions, test_size=0.5, random_state=42)

train_df = df_split[df_split['lesion_id'].isin(train_lesions)]
val_df   = df_split[df_split['lesion_id'].isin(val_lesions)]
test_df  = df_split[df_split['lesion_id'].isin(test_lesions)]

print(f"Train : {len(train_df)} images")
print(f"Val   : {len(val_df)} images")
print(f"Test  : {len(test_df)} images")
print(f"Total : {len(train_df)+len(val_df)+len(test_df)} (should be 23257)")

print("\nClass distribution in train:")
print(train_df[label_cols].sum().astype(int).sort_values(ascending=False))

# Save all three splits
train_df.to_csv("../data/layer0_train.csv", index=False)
val_df.to_csv("../data/layer0_val.csv", index=False)
test_df.to_csv("../data/layer0_test.csv", index=False)
print("\nSaved: layer0_train.csv, layer0_val.csv, layer0_test.csv")

Train : 18707 images
Val   : 2244 images
Test  : 2306 images
Total : 23257 (should be 23257)

Class distribution in train:
NV      9208
MEL     3351
BCC     2721
BKL     1783
AK       711
SCC      526
DF       206
VASC     201
dtype: int32

Saved: layer0_train.csv, layer0_val.csv, layer0_test.csv
